In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [3]:
load_dotenv()

True

In [4]:
model = ChatGroq(
    model='openai/gpt-oss-20b'
)

In [7]:
class SentimentSchema(BaseModel):

    sentiment: Literal['Positive', 'Negative'] = Field(description='Sentiment of the review')

In [8]:
structured_model = model.with_structured_output(SentimentSchema)

In [9]:
prompt = 'What is the sentiment of the following review - the software is too bad'
structured_model.invoke(prompt)

SentimentSchema(sentiment='Negative')

In [10]:
class ReviewState(TypedDict):

    review : str
    sentiment : Literal['Positive', 'Negative']
    diagnosis : dict
    response :str

In [ ]:
def fina_sentiment(state: ReviewState):
    prompt = f'For the following review find out the sentiment /n {state['review']}'

    sentiment = structured_model.invoke(prompt).sentiment

    return{'sentiment':sentiment}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', fina_sentiment)